In [1]:
%load_ext autoreload
%autoreload 2

# Automatic Fix AV classification

This notebook demonstrates how to fix a predicted AV classification map using topological recontrustion of the vascular tree.


In [2]:
from pathlib import Path

import numpy as np
from fundus_data_toolkit.functional import open_image
from jppype import Mosaic, vscode_theme

from fundus_odmac_toolkit.models.segmentation import segment
from fundus_toolkits import FundusData
from fundus_vessels_toolkit.pipelines import AVSegToTree, GNNAVSegToTree, NaiveAVSegToTree
from fundus_vessels_toolkit.segment_to_graph.av_map_fixing import fix_av_map
from fundus_vessels_toolkit.utils.data_io import load_label_image
from fundus_vessels_toolkit.utils.jppype import draw_graph, draw_tree, draw_trees

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

### Load a fundus image and its AV map


In [3]:
IMG = Path("g_004.png")
PATH = Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/GAVE-train/")


# Path to the raw fundus image
RAW_PATH = PATH / "1-images" / IMG

# Path to the artery/vein segmentation
AV = PATH / "2-av" / IMG
TOPO = PATH / "3-topo" / IMG.with_suffix("")

fundus = FundusData(image=RAW_PATH, av=AV)
od_mac = segment(open_image(RAW_PATH)).numpy(force=True).argmax(axis=0)
fundus = fundus.update(od=od_mac == 1, macula=od_mac == 2, reshape_method="resize")

In [4]:
from fundus_vessels_toolkit.vascular_data_objects.vtree import VTree

av2tree = NaiveAVSegToTree()

trees = VTree.load(str(TOPO) + "_art.npz"), VTree.load(str(TOPO) + "_vei.npz")
fundus_fixed = fundus.update(
    av=fix_av_map(fundus.av, trees, force_initial_segmentation=False, expand_labels_by=1, discard_av=False)
)

m = Mosaic(3, cell_height=600)
fundus.draw(view=m[0])
draw_trees(av2tree(fundus), m[0])
fundus_fixed.draw(view=m[1])
draw_trees(trees, m[1])
fundus_fixed.draw(view=m[2])
m

[ WARN:0@7.849] global loadsave.cpp:1671 imencodeWithMetadata Unsupported depth image for selected encoder is fallbacked to CV_8U.


GridBox(children=(View2D(linkedTransformGroup='310408566f6f4b9ab25c10b58ac18d68'), View2D(linkedTransformGroup…

In [ ]:
from fundus_vessels_toolkit.models.metrics.topological import (
    valid_path_ratio,
    valid_path_ratio_cpp,
    sample_valid_path_ratio,
)

fundus_art = fundus.av == 1
fundus_fixed_art = fundus_fixed.av == 1

cor_art, inf_art = valid_path_ratio(fundus_art, fundus_fixed_art, skeletonize=True)
cor_vei, inf_vei = valid_path_ratio(fundus.av == 2, fundus_fixed.av == 2, skeletonize=True)
print(f"Arteries: {cor_art:.1%} correct paths, {1 - inf_art:.1%} incorrect paths")
print(f"Veins: {cor_vei:.1%} correct paths, {1 - inf_vei:.1%} incorrect paths")

(cor_art, n_art), (inf_art, n_fixed_art) = valid_path_ratio_cpp(fundus_art, fundus_fixed_art, skeletonize=True)
(cor_vei, n_vei), (inf_vei, n_fixed_vei) = valid_path_ratio_cpp(fundus.av == 2, fundus_fixed.av == 2, skeletonize=True)
print(f"Arteries: {cor_art:.1%} correct paths, {1 - inf_art:.1%} incorrect paths")
print(f"Veins: {cor_vei:.1%} correct paths, {1 - inf_vei:.1%} incorrect paths")

8.2 ms ± 77 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
Arteries: 82.3% correct paths, 9.5% incorrect paths
Veins: 83.4% correct paths, 13.3% incorrect paths
5.28 ms ± 50.3 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
Arteries: 82.3% correct paths, 9.5% incorrect paths
Veins: 83.4% correct paths, 13.3% incorrect paths


In [11]:
n_art, n_fixed_art

(8370700, 26022303)

In [10]:
valid_ratio_art = sample_valid_path_ratio(fundus_art, fundus_fixed_art)
valid_ratio_vei = sample_valid_path_ratio(fundus.av == 2, fundus_fixed.av == 2)
print(f"Arteries: {valid_ratio_art[0]:.1%} correct paths")
print(f"Veins: {valid_ratio_vei[0]:.1%} correct paths")

RuntimeError: expected scalar type Int but found UInt32

## Parse tree on the GT


Parse the topology of the ground truth segmentation and generate its topology map.


In [ ]:
seg2tree = NaiveAVSegToTree()
trees_gt = seg2tree(fundus_gt)

topo_maps = [rasterize_tree_topology(tree, expand_labels_by=10) for tree in trees_gt]
(art_labels, art_topo), (vei_labels, vei_topo) = topo_maps

- The `labels` indicate to which subtree each vessel pixel belong, as well as the branching patterns to reach it.
- The `topo` map monotonically increases with the distance from the subtree root.


In [ ]:
m = Mosaic(
    (2, 3),
    cols_titles=["VTree", "Branch labels", "Topology map"],
    rows_titles=["Art.", "Vein"],
    cell_height=400,
)
fundus_gt.draw(view=m[0, 0])
draw_tree(trees_gt[0], view=m[0, 0], artery=True, edge_labels=True)
m[0, 1].add_image(TopologicalLabel.map_to_rgb(art_labels))
m[0, 2].add_image(np.repeat(art_topo[:, :, None], 3, axis=2))
fundus_gt.draw(view=m[1, 0])
draw_tree(trees_gt[1], view=m[1, 0], artery=False, edge_labels=True)
m[1, 1].add_image(TopologicalLabel.map_to_rgb(vei_labels))
m[1, 2].add_image(np.repeat(vei_topo[:, :, None], 3, axis=2))
m

## Parse graph on the Prediction


In [ ]:
trees_fixed = AVSegToTree()(fundus)

fundus_fixed = fundus.update(vessels=fix_av_map(fundus.av, trees_fixed))

m = Mosaic(2, cols_titles=["Predicted", "Corrected"], cell_height=800)
fundus.draw(view=m[0])
fundus_fixed.draw(view=m[1])

draw_trees(trees_fixed, view=m[1])
m